## Generate Swaption Prices from HW Parameters

In [ ]:
import QuantLib as ql
from pydoe import lhs
import math
import pandas as pd
import random as rd
from py_vollib.black_scholes.implied_volatility import implied_volatility
from tqdm.notebook import tqdm, trange
import numpy as np
import os
import multiprocessing as mp

In [ ]:
def price_swaptions_hw(flat_rate, hw_params, swaption_details):
    # Unpacking the HW model parameters
    a, sigma = hw_params

    # Define evaluation date
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today

    # Construct the interest rate term structure
    rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, 
                                                             flat_rate, 
                                                             ql.Actual365Fixed()))
    index = ql.Euribor1Y(rate_handle)
    fixed_leg_tenor = ql.Period(1, ql.Years)
    fixed_leg_daycounter = ql.Actual360()
    floating_leg_daycounter = ql.Actual360()
    
    # Define the HW model
    model = ql.HullWhite(rate_handle, a, sigma)

    # List to store swaption prices
    swaption_prices = []

    # Pricing each swaption
    for expiry, tenor in swaption_details:
        vol_handle = ql.QuoteHandle(ql.SimpleQuote(0.10))
        helper = ql.SwaptionHelper(ql.Period(expiry, ql.Years),
                                   ql.Period(tenor, ql.Years),
                                   vol_handle,
                                   index,
                                   fixed_leg_tenor,
                                   fixed_leg_daycounter,
                                   floating_leg_daycounter,
                                   rate_handle
                                   )

        engine = ql.JamshidianSwaptionEngine(model)
        helper.setPricingEngine(engine)

        # Store the NPV (price) of the swaption
        swaption_prices.append(helper.modelValue())

    return swaption_prices

In [ ]:
flat_rate = 0.05 
hw_params = [0.2, 0.025]  
swaption_details = [[1, 5], [2, 10]]
prices = price_swaptions_hw(flat_rate, hw_params, swaption_details)
print(prices)

In [ ]:
flat_rate = 0.01
g2pp_params = [0.3, 0.03] 
swaption_details = [[1, 5], [2, 10]]
prices = price_swaptions_hw(flat_rate, hw_params, swaption_details)
print(prices)

In [ ]:
# Function to generate LHS samples
def generate_lhs_samples(samples, dimensions):
    return lhs(dimensions, samples)

# Function to scale LHS samples according to the specified bounds
def scale_lhs_samples(lhs_samples, bounds):
    return np.array([sample * (bound[1] - bound[0]) + bound[0] for sample, bound in zip(lhs_samples.T, bounds)]).T

In [ ]:
def generateData(ID, pkgsize, filename):
    np.random.seed(ID)
    bounds = [(0.0, 0.5), (0.0, 0.05), (0.0, 0.1)]
    option_data = []
    lhs_samples = generate_lhs_samples(pkgsize, len(bounds))
    scaled_samples = scale_lhs_samples(lhs_samples, bounds)
    swaptions_defs = [[1, 5], [2, 4], [3, 3], [4, 2], [5, 1]]
    
    for i in range(pkgsize):
        a, sigma, rate = scaled_samples[i]
        hw_params = [a, sigma]
        swaption_prices = price_swaptions_hw(flat_rate, hw_params, swaptions_defs)
        swaption_prices.extend(scaled_samples[i])
        
        option_data.append(swaption_prices)
    heading_list = ['1, 5', '2, 4', '3, 3', '4, 2', '5, 1', 'a', 'sigma', 'rate']
    df = pd.DataFrame(option_data, columns=heading_list)
    df.to_csv(filename + str(ID) + '.csv', index=False)
    return(ID)

In [ ]:
generateData(1, 10, 'TestHW')

In [ ]:
filename = 'HW_option_1_'
no_threads = os.cpu_count() - 4  # Adjusting the number of threads
pkgsize = 10000  # Number of samples per thread
dataSize = 100 # Total number of iterations/data chunks

In [ ]:
pool = mp.Pool(processes=no_threads)
pbar = tqdm(total=no_threads)
results = [pool.apply_async(generateData, args=(i, pkgsize, filename), callback=lambda _: pbar.update(1)) for i in range(dataSize)]
output = [p.get() for p in results]
pool.close()
pool.join()